# Convert Qwen3 4B Instruct for Triton vLLM

Run this notebook inside the Triton Control code-server workspace. It downloads `Qwen/Qwen3-4B-Instruct-2507`, places the model snapshot under `qwen3_4b_instruct/1/model`, and writes the `model.json` file used by Triton's vLLM backend.

The download is several GB. Use a workspace volume with enough free space before running all cells.

In [ ]:
%pip install huggingface_hub

In [ ]:
import json
from pathlib import Path

from huggingface_hub import snapshot_download

In [ ]:
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
MODEL_ROOT = Path("qwen3_4b_instruct/1")
MODEL_DIR = MODEL_ROOT / "model"
MODEL_JSON = MODEL_ROOT / "model.json"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
print(f"Preparing {MODEL_ID} under {MODEL_DIR}")

In [ ]:
snapshot_download(
    repo_id=MODEL_ID,
    local_dir=MODEL_DIR,
    ignore_patterns=[".git*", "*.ckpt", "*.h5", "*.msgpack", "*.onnx"],
)

print(f"Downloaded model snapshot to {MODEL_DIR}")

In [ ]:
engine_config = {
    "model": "./model",
    "tokenizer": "./model",
    "dtype": "float16",
    "max_model_len": 8192,
    "gpu_memory_utilization": 0.85,
    "enforce_eager": True,
}

MODEL_JSON.write_text(json.dumps(engine_config, indent=2) + "\n", encoding="utf-8")
print(MODEL_JSON.read_text(encoding="utf-8"))

After the notebook finishes, deploy the `qwen3-4b-instruct-vllm` folder with the Triton Control Deploy extension. The extension detects `backend: "vllm"` and uses the vLLM repository sync path automatically.